# Modulo 6 · Lezione 6 — pandas per l'analisi dei dati

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Series e DataFrame |
| 2 | Creazione, importazione e ispezione |
| 3 | Selezione e indicizzazione |
| 4 | Filtraggio e trasformazione |
| 5 | Raggruppamento e aggregazione |
| 6 | Pulizia dei dati (NaN, duplicati, tipi) |
| 7 | Merge, pivot e visualizzazione di base |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 100)
print(f"pandas {pd.__version__}")

---
## Motivazione

```
NumPy è perfetto per array numerici omogenei.
Ma i dati reali sono:
  - Misti: numeri, testo, date, booleani
  - Sporchi: valori mancanti, duplicati, tipi errati
  - Tabulari: righe (osservazioni) × colonne (variabili)

pandas = NumPy + indice + colonne con nome + tipi misti
                + operazioni di database + pulizia

Prima di addestrare qualsiasi modello → i tuoi dati passano per pandas.
```

---
## 1. Series — l'array con indice

In [ ]:
# Serie da lista
notas = pd.Series([8.5, 7.0, 9.2, 6.1, 8.8], name="voto_finale")
print("Serie semplice:")
print(notas)

# Serie con indice personalizzato
notas_nombres = pd.Series(
    [8.5, 7.0, 9.2, 6.1, 8.8],
    index=["Ana", "Bruno", "Carla", "Diego", "Elena"],
    name="voto_finale"
)
print("\nSerie con indice di nomi:")
print(notas_nombres)

# Operazioni vettorizzate (come NumPy)
print(f"\nMedia: {notas_nombres.mean():.2f}")
print(f"Promossi (≥5): {(notas_nombres >= 5).sum()}")
print(f"\nVoto normalizzato (0-1):")
print(((notas_nombres - notas_nombres.min()) / (notas_nombres.max() - notas_nombres.min())).round(3))

# Accesso per etichetta e indice
print(f"\nVoto di Carla (per nome): {notas_nombres['Carla']}")
print(f"Primo elemento (per pos): {notas_nombres.iloc[0]}")

---
## 2. DataFrame — la tabella di dati

In [ ]:
# Creare da dizionario
np.random.seed(42)
n = 100

df = pd.DataFrame({
    'id':          range(1, n+1),
    'nombre':      [f"Est_{i:03d}" for i in range(1, n+1)],
    'edad':        np.random.randint(18, 35, n),
    'ciudad':      np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'], n),
    'horas_estudio': np.random.normal(5, 1.5, n).clip(0).round(1),
    'nota_teoria': np.random.normal(6.5, 1.5, n).clip(0, 10).round(1),
    'nota_practica': np.random.normal(7.0, 1.2, n).clip(0, 10).round(1),
    'aprobado':    np.random.choice([True, False], n, p=[0.75, 0.25]),
})
df['nota_final'] = (df['nota_teoria'] * 0.4 + df['nota_practica'] * 0.6).round(1)

print("=== Prime righe ===")
print(df.head())

print("\n=== Informazioni sul DataFrame ===")
df.info()

print("\n=== Statistiche descrittive ===")
print(df.describe().round(2))

print(f"\nDimensioni: {df.shape}  ({df.shape[0]} righe × {df.shape[1]} colonne)")

---
## 3. Selezione e indicizzazione

In [ ]:
# Colonne
print("Una colonna:")
print(df['ciudad'].head())

print("\nPiù colonne:")
print(df[['nombre', 'nota_final', 'aprobado']].head())

# .loc  — per etichette (indice + nome colonna)
print("\n.loc riga 0, colonne nota_teoria e nota_practica:")
print(df.loc[0, ['nota_teoria', 'nota_practica']])

# .iloc — per posizione
print("\n.iloc prime 3 righe, colonne 0,2,4:")
print(df.iloc[:3, [0, 2, 4]])

# Slicing
print("\nRighe da 5 a 8 (incluse), tutte le colonne:")
print(df.loc[5:8])

# Riepilogo delle differenze
print("\n=== Riferimento rapido ===")
print("df['col']              → colonna come Serie")
print("df[['c1','c2']]        → sottoinsieme di colonne")
print("df.loc[i, 'col']       → cella per etichetta")
print("df.iloc[i, j]          → cella per posizione")
print("df.loc[2:5, 'col']     → intervallo di righe, colonna")

---
## 4. Filtraggio e trasformazione

In [ ]:
# Filtraggio booleano
aprobados = df[df['aprobado'] == True]
print(f"Promossi: {len(aprobados)}")

# Condizioni multiple
buenos_madrid = df[(df['ciudad'] == 'Madrid') & (df['nota_final'] >= 8)]
print(f"Voto ≥ 8 a Madrid: {len(buenos_madrid)}")

# .isin() per valori multipli
levante = df[df['ciudad'].isin(['Barcelona', 'Valencia'])]
print(f"Barcelona o Valencia: {len(levante)}")

# .query() — sintassi più leggibile
resultado = df.query("nota_final >= 8 and edad < 25")
print(f"Voto ≥ 8 ed età < 25: {len(resultado)}")

# Aggiungere colonne calcolate
df['rendimiento'] = df['nota_final'] / df['horas_estudio']
df['categoria']   = pd.cut(df['nota_final'],
                            bins=[0, 4.9, 6.9, 8.9, 10],
                            labels=['Bocciato', 'Sufficiente', 'Buono', 'Ottimo'])

print("\nDistribuzione per categoria:")
print(df['categoria'].value_counts())

# .apply() per funzioni personalizzate
def clasificar_edad(edad):
    if edad < 22: return 'junior'
    elif edad < 28: return 'mid'
    else: return 'senior'

df['grupo_edad'] = df['edad'].apply(clasificar_edad)
print("\nGruppi di età:")
print(df['grupo_edad'].value_counts())

---
## 5. Raggruppamento e aggregazione

In [ ]:
# Statistiche per città
por_ciudad = df.groupby('ciudad')['nota_final'].agg(['mean', 'std', 'count'])
por_ciudad.columns = ['media', 'std', 'n']
por_ciudad = por_ciudad.round(3)
print("Voti per città:")
print(por_ciudad)

# Colonne e statistiche multiple
print("\nRiepilogo per categoria:")
resumen = df.groupby('categoria').agg(
    n=('nota_final', 'count'),
    nota_media=('nota_final', 'mean'),
    horas_media=('horas_estudio', 'mean'),
    pct_aprobados=('aprobado', 'mean')
).round(3)
print(resumen)

# Raggruppamento multiplo
print("\nMedia del voto per città e gruppo_età:")
multi = df.groupby(['ciudad', 'grupo_edad'])['nota_final'].mean().round(2).unstack()
print(multi)

# transform: aggiungere statistica di gruppo come colonna
df['media_ciudad'] = df.groupby('ciudad')['nota_final'].transform('mean')
df['sobre_media_ciudad'] = df['nota_final'] > df['media_ciudad']
print(f"\nStudenti sopra la media della propria città: {df['sobre_media_ciudad'].sum()}")

---
## 6. Pulizia dei dati

In [ ]:
# Dataset con problemi
datos_sucios = pd.DataFrame({
    'nombre':  ['Ana', 'Bruno', 'Carla', 'Diego', 'Ana', 'Elena', None, 'Fran'],
    'edad':    [25, None, 30, 22, 25, 28, 31, -5],
    'nota':    [8.5, 7.0, None, 6.1, 8.5, None, 9.2, 7.8],
    'ciudad':  ['Madrid', 'madrid', 'BARCELONA', 'Valencia', 'Madrid', 'Sevilla', 'Barcelona', 'Madrid'],
    'email':   ['ana@mail.com', 'bruno@', 'carla@mail.com', 'diego@mail.com',
                'ana@mail.com', 'elena@mail.com', None, 'fran@mail.com'],
})

print("Dataset con problemi:")
print(datos_sucios)

# 1. Valori mancanti
print("\n=== Valori mancanti ===")
print(datos_sucios.isnull().sum())
print(f"Totale: {datos_sucios.isnull().sum().sum()}")

# Riempire / eliminare
df_clean = datos_sucios.copy()
df_clean['nota'].fillna(df_clean['nota'].median(), inplace=True)
df_clean['edad'].fillna(df_clean['edad'].median(), inplace=True)
df_clean.dropna(subset=['nombre'], inplace=True)  # eliminare righe senza nome

# 2. Duplicati
print("\n=== Duplicati ===")
print(f"Righe duplicate: {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates()

# 3. Normalizzare il testo
df_clean['ciudad'] = df_clean['ciudad'].str.strip().str.title()
df_clean['nombre'] = df_clean['nombre'].str.strip().str.title()

# 4. Valori impossibili
print(f"\nEtà negative: {(df_clean['edad'] < 0).sum()}")
df_clean = df_clean[df_clean['edad'] > 0]

# 5. Tipi di dato
df_clean['edad'] = df_clean['edad'].astype(int)

print("\nDataset pulito:")
print(df_clean.reset_index(drop=True))

---
## 7. Merge, pivot e visualizzazione

In [ ]:
# Merge (equivalente al JOIN di SQL)
estudiantes = pd.DataFrame({
    'id_est': [1, 2, 3, 4, 5],
    'nombre': ['Ana', 'Bruno', 'Carla', 'Diego', 'Elena'],
    'id_ciudad': [10, 10, 20, 30, 20]
})
ciudades = pd.DataFrame({
    'id_ciudad': [10, 20, 30],
    'ciudad': ['Madrid', 'Barcelona', 'Valencia'],
    'region': ['Centro', 'Catalogna', 'Comunità Valenciana']
})

combinado = pd.merge(estudiantes, ciudades, on='id_ciudad', how='left')
print("Merge (left join) studenti + città:")
print(combinado)

# Pivot table
print("\nTabella pivot: media del voto per città e categoria:")
pivot = df.pivot_table(
    values='nota_final',
    index='ciudad',
    columns='grupo_edad',
    aggfunc='mean'
).round(2)
print(pivot)

# Visualizzazione con pandas
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Istogramma
df['nota_final'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribuzione del voto finale')
axes[0].set_xlabel('Voto')

# Barre per città
df.groupby('ciudad')['nota_final'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title('Media voto per città')
axes[1].set_xlabel('Voto medio')

# Boxplot per categoria
df.boxplot(column='horas_estudio', by='categoria', ax=axes[2],
           flierprops=dict(marker='o', ms=4, alpha=0.5))
axes[2].set_title('Ore di studio per categoria')
plt.suptitle('')

plt.tight_layout(); plt.show()

---
## Pratica

### Esercizio 1 — Esplorazione iniziale

**Consegna:** Crea questo dataset di vendite di un negozio ed esplorarlo:

```python
np.random.seed(1)
n = 200
ventas = pd.DataFrame({
    'fecha':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'producto':  np.random.choice(['laptop','tablet','phone','smartwatch'], n),
    'cantidad':  np.random.randint(1, 20, n),
    'precio':    np.random.choice([299, 499, 799, 1299], n),
    'descuento': np.random.uniform(0, 0.3, n).round(2),
    'vendedor':  np.random.choice(['Alice','Bob','Carol','Dave'], n),
})
ventas['total'] = (ventas['cantidad'] * ventas['precio'] * (1 - ventas['descuento'])).round(2)
```

1. `shape`, `dtypes`, `info()`, `describe()`
2. Quante vendite ci sono per ogni prodotto?
3. Quale venditore ha il fatturato totale più alto?
4. Filtra le vendite con sconto > 20% e totale > 5000€
5. Aggiungi una colonna `mese` estraendo il mese dalla `data`

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import pandas as pd
import numpy as np

np.random.seed(1)
n = 200
ventas = pd.DataFrame({
    'fecha':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'producto':  np.random.choice(['laptop','tablet','phone','smartwatch'], n),
    'cantidad':  np.random.randint(1, 20, n),
    'precio':    np.random.choice([299, 499, 799, 1299], n),
    'descuento': np.random.uniform(0, 0.3, n).round(2),
    'vendedor':  np.random.choice(['Alice','Bob','Carol','Dave'], n),
})
ventas['total'] = (ventas['cantidad'] * ventas['precio'] * (1 - ventas['descuento'])).round(2)

# 1.
print(f"Shape: {ventas.shape}")
print(ventas.dtypes)
ventas.info()
print(ventas.describe().round(2))

# 2.
print("\nVendite per prodotto:")
print(ventas['producto'].value_counts())

# 3.
facturacion = ventas.groupby('vendedor')['total'].sum().sort_values(ascending=False)
print(f"\nFatturato per venditore:")
print(facturacion)
print(f"\nMiglior venditore: {facturacion.index[0]} ({facturacion.iloc[0]:,.2f}€)")

# 4.
filtro = ventas[(ventas['descuento'] > 0.20) & (ventas['total'] > 5000)]
print(f"\nVendite con sconto >20% e totale >5000€: {len(filtro)}")
print(filtro[['producto', 'cantidad', 'precio', 'descuento', 'total']].head())

# 5.
ventas['mes'] = ventas['fecha'].dt.month
print(f"\nVendite per mese:")
print(ventas.groupby('mes')['total'].sum().round(2))
```
</details>

---
### Esercizio 2 — Pulizia completa

**Consegna:** Pulisci questo dataset con problemi multipli:

```python
datos = pd.DataFrame({
    'id':     [1, 2, 3, 4, 5, 6, 7, 2],          # duplicato
    'nombre': ['Ana', 'Bruno', None, 'Diego', 'ELENA', 'fran', 'Gema', 'Bruno'],
    'edad':   [25, 28, 999, 22, 30, -3, None, 28],  # età impossibili
    'salario':[45000, None, 38000, 52000, 61000, 29000, None, 47000],
    'dpto':   ['IT', 'IT', 'HR', 'hr', 'Finance', 'FINANCE', 'IT', 'IT'],
    'activo': ['Si', 'No', 'si', 'SI', 'no', 'NO', 'Si', 'No'],
})
```

1. Individua tutti i problemi
2. Elimina i duplicati (mantieni il primo)
3. Pulisci il testo (nome, dpto, attivo → True/False)
4. Correggi le età impossibili → NaN
5. Imputa gli stipendi con la mediana del dipartimento

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import pandas as pd
import numpy as np

datos = pd.DataFrame({
    'id':     [1, 2, 3, 4, 5, 6, 7, 2],
    'nombre': ['Ana', 'Bruno', None, 'Diego', 'ELENA', 'fran', 'Gema', 'Bruno'],
    'edad':   [25, 28, 999, 22, 30, -3, None, 28],
    'salario':[45000, None, 38000, 52000, 61000, 29000, None, 47000],
    'dpto':   ['IT', 'IT', 'HR', 'hr', 'Finance', 'FINANCE', 'IT', 'IT'],
    'activo': ['Si', 'No', 'si', 'SI', 'no', 'NO', 'Si', 'No'],
})

df = datos.copy()

# 1. Diagnosi
print("Problemi rilevati:")
print(f"  Righe duplicate: {df.duplicated().sum()}")
print(f"  Nulli:\n{df.isnull().sum()}")
print(f"  Età impossibili (>120 o <0): {((df['edad']>120)|(df['edad']<0)).sum()}")

# 2. Eliminare duplicati
df = df.drop_duplicates(subset='id', keep='first')

# 3. Pulire il testo
df['nombre'] = df['nombre'].str.strip().str.title()
df['dpto']   = df['dpto'].str.strip().str.upper()
df['activo'] = df['activo'].str.strip().str.lower().map({'si': True, 'no': False})

# 4. Età impossibili → NaN
df['edad'] = df['edad'].where((df['edad'] >= 16) & (df['edad'] <= 80), other=np.nan)

# 5. Imputare lo stipendio con la mediana del dpto
mediana_dpto = df.groupby('dpto')['salario'].transform('median')
df['salario'] = df['salario'].fillna(mediana_dpto)

print("\nDataset pulito:")
print(df.to_string(index=False))
print(f"\nNulli restanti: {df.isnull().sum().sum()}")
```
</details>

---
### Esercizio 3 — Analisi dei raggruppamenti

**Consegna:** Usando il DataFrame `df` generato all'inizio della lezione:

1. Media, massimo e minimo di `nota_final` per `ciudad` e `grupo_edad`
2. Quale città ha il tasso di promossi più alto?
3. Crea una colonna `ranking` con il rango del voto all'interno di ogni città (1 = voto migliore)
4. Top 3 studenti per città in base al voto finale

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
# 1.
stats_ciudad_edad = df.groupby(['ciudad', 'grupo_edad'])['nota_final'].agg(['mean','max','min']).round(2)
print("Voto finale per città e gruppo di età:")
print(stats_ciudad_edad)

# 2.
tasa_aprobados = df.groupby('ciudad')['aprobado'].mean().sort_values(ascending=False)
print(f"\nTasso di promossi per città:")
print(tasa_aprobados.round(3))
print(f"\nCittà con il tasso più alto: {tasa_aprobados.index[0]}")

# 3. Ranking all'interno della città
df['ranking_ciudad'] = df.groupby('ciudad')['nota_final'].rank(ascending=False, method='min').astype(int)
print("\nPrimi record con ranking:")
print(df[['nombre', 'ciudad', 'nota_final', 'ranking_ciudad']].head(10))

# 4. Top 3 per città
print("\nTop 3 per città:")
top3 = df[df['ranking_ciudad'] <= 3].sort_values(['ciudad', 'ranking_ciudad'])
print(top3[['ciudad', 'ranking_ciudad', 'nombre', 'nota_final']].to_string(index=False))
```
</details>

---
### Esercizio 4 — Merge e pivot

**Consegna:** Combina questi due DataFrame e crea analisi incrociate:

```python
np.random.seed(10)
cursos_df = pd.DataFrame({
    'id_curso': range(1, 6),
    'nombre_curso': ['Python', 'ML', 'Deep Learning', 'SQL', 'Cloud'],
    'nivel': ['basico', 'intermedio', 'avanzado', 'basico', 'intermedio'],
    'duracion_h': [20, 40, 60, 15, 30]
})
inscripciones_df = pd.DataFrame({
    'id_est': np.random.randint(1, 21, 80),
    'id_curso': np.random.randint(1, 6, 80),
    'nota': np.random.normal(6.5, 1.5, 80).clip(0, 10).round(1),
    'completado': np.random.choice([True, False], 80, p=[0.8, 0.2])
})
```

1. Fai il merge tra iscrizioni e corsi
2. Voto medio per corso (ordinato)
3. Tabella pivot: voto medio × livello × completato
4. Quale livello ha il tasso di completamento più alto?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import pandas as pd
import numpy as np

np.random.seed(10)
cursos_df = pd.DataFrame({
    'id_curso': range(1, 6),
    'nombre_curso': ['Python', 'ML', 'Deep Learning', 'SQL', 'Cloud'],
    'nivel': ['basico', 'intermedio', 'avanzado', 'basico', 'intermedio'],
    'duracion_h': [20, 40, 60, 15, 30]
})
inscripciones_df = pd.DataFrame({
    'id_est': np.random.randint(1, 21, 80),
    'id_curso': np.random.randint(1, 6, 80),
    'nota': np.random.normal(6.5, 1.5, 80).clip(0, 10).round(1),
    'completado': np.random.choice([True, False], 80, p=[0.8, 0.2])
})

# 1. Merge
df_m = pd.merge(inscripciones_df, cursos_df, on='id_curso', how='left')
print("Merged (prime righe):")
print(df_m.head())

# 2. Voto medio per corso
print("\nVoto medio per corso:")
nota_curso = df_m.groupby('nombre_curso')['nota'].mean().sort_values(ascending=False).round(3)
print(nota_curso)

# 3. Pivot: livello × completato
print("\nVoto medio per livello e completato:")
pivot = df_m.pivot_table(values='nota', index='nivel', columns='completado', aggfunc='mean').round(3)
pivot.columns = ['Non completato', 'Completato']
print(pivot)

# 4. Tasso di completamento per livello
tasa = df_m.groupby('nivel')['completado'].mean().sort_values(ascending=False)
print(f"\nTasso di completamento per livello:")
print(tasa.round(3))
print(f"Livello con il tasso più alto: {tasa.index[0]}")
```
</details>

---
### Esercizio 5 (sfida) — Pipeline di preprocessing

**Consegna:** Implementa una funzione `preprocesar(df)` che riceve un DataFrame grezzo e restituisce uno pulito e arricchito, applicando in ordine:

1. Eliminare i duplicati
2. Normalizzare il testo (colonne di tipo object)
3. Correggere gli outlier numerici (sostituisci con NaN i valori fuori da Q1-3×IQR, Q3+3×IQR)
4. Imputare i NaN numerici con la mediana della propria colonna
5. Aggiungere la colonna `calidad_dato` = percentuale di campi non nulli che aveva il record originale
6. Restituisce (df_pulito, dizionario_report) con le statistiche del processo

Provala con il dataset delle vendite dell'esercizio 1.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import pandas as pd
import numpy as np

def preprocesar(df_raw):
    df = df_raw.copy()
    reporte = {}

    # Qualità originale (prima di qualsiasi modifica)
    calidad_original = df.notna().mean(axis=1) * 100

    # 1. Duplicati
    antes = len(df)
    df = df.drop_duplicates()
    reporte['filas_eliminadas_dup'] = antes - len(df)

    # 2. Normalizzare il testo
    cols_obj = df.select_dtypes(include='object').columns
    for col in cols_obj:
        df[col] = df[col].str.strip().str.title()
    reporte['cols_texto_normalizadas'] = list(cols_obj)

    # 3. Outlier → NaN (usando 3×IQR, più conservativo di 1.5×IQR)
    cols_num = df.select_dtypes(include=np.number).columns
    outliers_total = 0
    for col in cols_num:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        mask = (df[col] < Q1 - 3*IQR) | (df[col] > Q3 + 3*IQR)
        n_out = mask.sum()
        outliers_total += n_out
        df.loc[mask, col] = np.nan
    reporte['outliers_convertidos_nan'] = outliers_total

    # 4. Imputare i NaN numerici
    nan_antes = df[cols_num].isnull().sum().sum()
    for col in cols_num:
        df[col].fillna(df[col].median(), inplace=True)
    reporte['nan_imputados'] = nan_antes

    # 5. Qualità del dato
    df['calidad_dato'] = calidad_original.reindex(df.index).fillna(0).round(1)

    reporte['filas_finales']   = len(df)
    reporte['columnas_finales'] = len(df.columns)

    return df, reporte

# Prova con le vendite
np.random.seed(1)
n = 200
ventas = pd.DataFrame({
    'fecha':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'producto':  np.random.choice(['laptop','tablet','phone','smartwatch'], n),
    'cantidad':  np.random.randint(1, 20, n),
    'precio':    np.random.choice([299, 499, 799, 1299], n),
    'descuento': np.random.uniform(0, 0.3, n).round(2),
    'vendedor':  np.random.choice(['Alice','Bob','Carol','Dave'], n),
})
ventas['total'] = (ventas['cantidad'] * ventas['precio'] * (1 - ventas['descuento'])).round(2)
# Aggiungere problemi artificiali
ventas.loc[0, 'cantidad'] = None
ventas.loc[1, 'precio']   = 99999  # outlier
ventas = pd.concat([ventas, ventas.iloc[[5]]])  # duplicato

df_limpio, reporte = preprocesar(ventas)
print("Report del preprocessing:")
for k, v in reporte.items():
    print(f"  {k}: {v}")
print(f"\nQualità media dei dati: {df_limpio['calidad_dato'].mean():.1f}%")
```
</details>

---
## Riepilogo

```python
import pandas as pd

# Creare ed esplorare
df = pd.DataFrame(diccionario)
df.info();  df.describe();  df.head()

# Selezionare
df['col'];  df[['c1','c2']]
df.loc[fila, col];  df.iloc[i, j]

# Filtrare
df[df['col'] > val]
df.query("col > val and otro == 'x'")

# Trasformare
df['nueva'] = df['a'] + df['b']
df['cat']   = pd.cut(df['num'], bins=[...])
df['res']   = df['col'].apply(mi_funcion)

# Raggruppare
df.groupby('col')['val'].agg(['mean','std','count'])
df.groupby('col')['val'].transform('mean')  # restituisce la stessa lunghezza

# Pulire
df.isnull().sum();  df.drop_duplicates()
df['col'].fillna(df['col'].median())
df.dropna(subset=['col_critica'])

# Combinare
pd.merge(df1, df2, on='key', how='left')
df.pivot_table(values='v', index='i', columns='c', aggfunc='mean')
```

## Cosa vediamo nella prossima lezione?
Nella **Lezione 7** chiudiamo il modulo con un'**analisi esplorativa completa** (EDA): carichiamo un dataset reale, applichiamo tutto ciò che abbiamo imparato nel modulo e ne traiamo conclusioni operative.